## Описание
**Applepen** $-$ это большая торговая сеть, которая занимается продажей всего двух продуктов: **яблок** и **карандашей**.
Ее магазины расположены в различных уголках Соединенных Штатов и более 10 лет обслуживают покупателей.

Недавно топ-менеджмент компании решил более активно использовать имеющиеся у них данные в принятии решений.
Каждый магазин собирает информацию о:

1. закупках (поставки яблок и карандашей два раза в месяц),
2. продажах (лог транзакций, по записи на каждую проданную позицию),
3. инвентарь (месячные данные общего количества яблок и карандашей на складе).

Данные доступны в формате CSV.
Внутри файла данные отсортированы ***по дате***.

К сожалению, данные никогда не консолидировались и не проверялись.
Вам необходимо получить следующие данные в CSV-файлах:

### 1. Состояние склада на каждый день
Данные о состоянии склада в конце каждого дня после того как все поставки и продажы были совершены.
Подобная информация будет очень ценна менеджерам магазинов.
Состояние склада должно строится на основне месячных данных об инвентаре.
Известно, что люди воруют из магазинов, но сейчас нет никакой возможности узнать объем сворованного товара.

Данные должны быть в следующем виде:

|date|apple|pen|
|---|---|---|
|2006-01-01|14105|770|
|...|||

### 2. Месячные данные о количестве сворованного товара

Данные должны быть в следующем виде:

|date|apple|pen|
|---|---|---|
|2006-01-31|4|4|
|2006-02-28|5|0|
|...|||


### 3. Агрегированные данные об объемах продаж и количестве сворованной продукции по штату и году

Данные должны быть в следующем виде:

|year|state|apple_sold|apple_stolen|pen_sold|pen_stolen|
|---|---|---|---|---|---|
|2006|AK|4|2|2|1
|2007|AL|5|0|6|2|
|...||||||

In [14]:
import pandas as pd

#список магазинов, которые нужно обработать
prefixes = ['MS-s1', 'MS-m1']

#списки для хранения промежуточных результатов
all_daily_inventory = []
all_monthly_stolen = []
all_aggregated_sales = []

In [15]:
for prefix in prefixes:
    state = prefix.split('-')[0]
    df_supply = pd.read_csv(prefix + '-supply.csv', parse_dates=['date'])
    df_sell = pd.read_csv(prefix + '-sell.csv', parse_dates=['date'])
    df_inv = pd.read_csv(prefix + '-inventory.csv', parse_dates=['date'])
    #считаем количество строк для каждого типа SKU (ap = apple, pe = pen)
    df_sell['type'] = df_sell['sku_num'].apply(lambda x: 'apple' if '-ap-' in x else 'pen')
    daily_sales = df_sell.groupby(['date', 'type']).size().unstack(fill_value=0).reset_index()
    #убеждаемся, что обе колонки есть в наличии
    for col in ['apple', 'pen']:
        if col not in daily_sales.columns:
            daily_sales[col] = 0
    daily_sales = daily_sales.rename(columns={'apple': 'apple_sold', 'pen': 'pen_sold'})
    #обработка поставок
    daily_supply = df_supply.groupby('date')[['apple', 'pen']].sum().reset_index()
    daily_supply = daily_supply.rename(columns={'apple': 'apple_sup', 'pen': 'pen_sup'})
    #создаем временную сетку (от начала поставок до последнего инвентаря)
    full_range = pd.date_range(start=df_supply['date'].min(), end=df_inv['date'].max(), freq='D')
    store_df = pd.DataFrame({'date': full_range})
    #объединяем продажи и поставки с календарем
    store_df = store_df.merge(daily_supply, on='date', how='left').fillna(0)
    store_df = store_df.merge(daily_sales, on='date', how='left').fillna(0)
    #цикл расчета инвентаря и краж (Reconciliation)
    inventory_lookup = df_inv.set_index('date')
    curr_apple, curr_pen = 0, 0
    for _, row in store_df.iterrows():
        d = row['date']
        #прогноз на конец дня - остаток + поставка - продажи
        curr_apple += (row['apple_sup'] - row['apple_sold'])
        curr_pen += (row['pen_sup'] - row['pen_sold'])
        #если сегодня день инвентаризации — считаем разницу (воровство)
        if d in inventory_lookup.index:
            actual_apple = inventory_lookup.loc[d, 'apple']
            actual_pen = inventory_lookup.loc[d, 'pen']
            stolen_apple = curr_apple - actual_apple
            stolen_pen = curr_pen - actual_pen
            all_monthly_stolen.append({
                'date': d,
                'apple': int(stolen_apple),
                'pen': int(stolen_pen),
                'state': state,
                'year': d.year
            })
            #устанавливаем реальный остаток
            curr_apple, curr_pen = actual_apple, actual_pen
        all_daily_inventory.append({
            'date': d,
            'apple': int(curr_apple),
            'pen': int(curr_pen)
        })
    store_df['year'] = store_df['date'].dt.year
    annual_sales = store_df.groupby('year')[['apple_sold', 'pen_sold']].sum().reset_index()
    annual_sales['state'] = state
    all_aggregated_sales.append(annual_sales)

In [16]:
#1)состояние склада на каждый день (сумма по всем магазинам)
res_task1 = pd.DataFrame(all_daily_inventory).groupby('date').sum().reset_index()
res_task1.to_csv('task1_daily_inventory.csv', index=False)

#2)ворованный товар (сумма по всем магазинам)
res_task2 = pd.DataFrame(all_monthly_stolen).groupby('date')[['apple', 'pen']].sum().reset_index()
res_task2.to_csv('task2_monthly_stolen.csv', index=False)

#3)агрегированные данные (продажи + воровство)
df_sales = pd.concat(all_aggregated_sales)
df_stolen = pd.DataFrame(all_monthly_stolen).groupby(['year', 'state'])[['apple', 'pen']].sum().reset_index()
df_stolen = df_stolen.rename(columns={'apple': 'apple_stolen', 'pen': 'pen_stolen'})

res_task3 = df_sales.merge(df_stolen, on=['year', 'state'])
res_task3 = res_task3[['year', 'state', 'apple_sold', 'apple_stolen', 'pen_sold', 'pen_stolen']]
res_task3.to_csv('task3_aggregated_data.csv', index=False)

### 3. Агрегированные данные об объемах продаж и количестве сворованной продукции по штату и году

Данные должны быть в следующем виде:

|year|state|apple_sold|apple_stolen|pen_sold|pen_stolen|
|---|---|---|---|---|---|
|2006|AK|4|2|2|1
|2007|AL|5|0|6|2|
|...||||||

In [17]:
res_task3.head()

,year,state,apple_sold,apple_stolen,pen_sold,pen_stolen
0,2006,MS,33930,96,3683,105
1,2007,MS,33608,87,3729,65
2,2008,MS,34071,86,3658,81
3,2009,MS,33671,97,3657,99
4,2010,MS,33806,94,3760,80


### 2. Месячные данные о количестве сворованного товара

Данные должны быть в следующем виде:

|date|apple|pen|
|---|---|---|
|2006-01-31|4|4|
|2006-02-28|5|0|
|...|||

In [18]:
res_task2.head()

,date,apple,pen
0,2006-01-31,8,11
1,2006-02-28,6,8
2,2006-03-31,7,5
3,2006-04-30,13,6
4,2006-05-31,4,10


### 1. Состояние склада на каждый день
Данные о состоянии склада в конце каждого дня после того как все поставки и продажы были совершены.
Подобная информация будет очень ценна менеджерам магазинов.
Состояние склада должно строится на основне месячных данных об инвентаре.
Известно, что люди воруют из магазинов, но сейчас нет никакой возможности узнать объем сворованного товара.

Данные должны быть в следующем виде:

|date|apple|pen|
|---|---|---|
|2006-01-01|14105|770|
|...|||

In [19]:
res_task1.head()

,date,apple,pen
0,2006-01-01,17246,1061
1,2006-01-02,16318,1010
2,2006-01-03,15315,962
3,2006-01-04,14355,893
4,2006-01-05,13432,833
